### Städtepartner – Lösung Schritt für Schritt

*30. Bundeswettbewerb Informatik (2011), 1. Runde, Aufgabe 5*

Viele Paare von Städten sind Partnerschaften eingegangen. In einem besonderen Jubiläumsjahr soll von jeder Städtepartnerschaft innerhalb der EU eine der beiden beteiligten Städte ein Fest für ihre Partnerstadt ausrichten. Damit der Aufwand für die ganzen Feste keine Stadt überfordert, wird für jede Stadt eine Obergrenze für die Anzahl der von ihr auszurichtenden Feste vereinbart. Aber können die Feste überhaupt so verteilt werden, dass alle Obergrenzen eingehalten werden?

Die Abbildung der Aufgabe zeigt einige Städte mit ihren Obergrenzen und die Partnerschaftsbeziehungen zwischen ihnen. Kannst du eine Lösung für dieses Beispiel finden?

#### Aufgabe 5

Entwirf und implementiere ein Verfahren, das eine gültige (d.h. die Obergrenzen einhaltende) Verteilung der Feste findet, falls eine solche existiert. Dein Verfahren muss so effizient sein, dass es auch das größere auf www.bundeswettbewerb-informatik.de abgelegte Beispiel löst.

#### Beispieldaten

- `beispieldaten/cities-small.txt` und `partnerships-small.txt`: das Beispiel aus der Abbildung (28 Städte, 54 Partnerschaften)
- `beispieldaten/cities.txt` und `partnerships.txt`: das größere Beispiel (1491 Städte, 2305 Partnerschaften)
- `cities*.txt`: je Zeile die Obergrenze und der Stadtname in Anführungszeichen, z. B. `1 "Wien"`
- `partnerships*.txt`: je Zeile zwei Städte, z. B. `"Wien", "Prag"`

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Graphen:** Städte sind Knoten, Partnerschaften sind Kanten. Ein Fest auszurichten heißt, eine Kante **zu richten**: Die Kante zeigt auf die Stadt, die das Fest ausrichtet. Dann darf in jede Stadt höchstens so viele Kanten zeigen, wie ihre Obergrenze erlaubt.
- **Breitensuche** (oder Tiefensuche): Von einem Knoten aus alle Knoten finden, die man über Kanten erreicht. Hier läuft sie **rückwärts** entlang der gerichteten Kanten.
- **Verbessernde Pfade:** Wenn etwas nicht passt, sucht man einen Weg, auf dem man die Last Schritt für Schritt weitergibt. Dasselbe Prinzip steckt hinter vielen Zuordnungs- und Flussverfahren.
- **Fluss in Netzwerken** (nur als Hintergrund, die Hinweise zeigen es als zweiten Ansatz): Die Aufgabe lässt sich auf einen maximalen Fluss zurückführen. Unser Verfahren ist im Kern ein solches Flussverfahren.
- **Mengen- und Zählargument:** Wenn eine Gruppe von Städten untereinander mehr Partnerschaften hat, als sie zusammen Feste ausrichten darf, gibt es sicher keine Lösung.

**Nützliche Funktionen und Bibliotheken**
- Dictionary (`{name: nummer}`): Städtenamen in Nummern umrechnen.
- `collections.deque` oder eine Liste mit Index als Warteschlange für die Breitensuche.
- `itertools.product([0, 1], repeat=m)`: alle Richtungen von `m` Kanten (nur für den Test an kleinen Graphen).
- `random`: Zufallsgraphen zum Testen.

## Die Idee in Worten

1. **Anfang:** Jede Partnerschaft bekommt irgendeinen Ausrichter, wir nehmen die zuerst genannte Stadt.
2. Jetzt zählen wir für jede Stadt `v`, wie viele Feste sie ausrichtet (`fest[v]`), und vergleichen mit der Obergrenze `grenze[v]`. Ist keine Stadt über der Grenze, sind wir fertig.
3. **Verbessern:** Sei `v` eine überlastete Stadt. Wir suchen eine Stadt `u`, die noch freie Plätze hat und von der aus man zu `v` kommt, wenn man den Kanten in ihrer Richtung folgt. Auf diesem Weg `u → … → v` **drehen wir jede Kante um**: Jede Stadt auf dem Weg gibt ein Fest ab und bekommt eins dafür, nur `v` gibt eines ab und `u` bekommt eines dazu. Die Städte dazwischen merken nichts. `v` ist jetzt um ein Fest entlastet.
4. Die Suche nach `u` läuft rückwärts von `v` aus. Findet sie keine Stadt mit freiem Platz, ist `v` mitsamt allen gefundenen Städten **nicht zu retten**: Es gibt keine Lösung.

Jede Verbesserung verringert die Summe der Überlastungen um 1. Mit `m` Partnerschaften gibt es also höchstens `m` Verbesserungen, jede kostet eine Suche über alle Kanten. Das sind höchstens etwa `m²` Schritte, für 2305 Partnerschaften also höchstens ein paar Millionen.

## Schritt 1: Daten einlesen

Wir rechnen Städtenamen in Nummern `0, 1, 2, …` um. `grenze[v]` ist die Obergrenze der Stadt `v`, `namen[v]` ihr Name. Eine Partnerschaft ist ein Paar `(a, b)` von Städtenummern, `paare` die Liste aller Partnerschaften. Die Zeilen sehen so aus: `1 "Wien"` und `"Wien", "Prag"`. Wir schneiden die Anführungszeichen ab und zerlegen die Zeile mit `split`.

In [ ]:
def lies_staedte(dateiname):
    namen = []
    grenze = []
    with open(dateiname, encoding='utf-8') as f:
        for zeile in f:
            zeile = zeile.strip()
            if zeile == '':
                continue
            zahl, name = zeile.split(' ', 1)         # "1 "Wien"" -> "1" und ""Wien""
            grenze.append(int(zahl))
            namen.append(name.strip('"'))
    return namen, grenze


def lies_partnerschaften(dateiname, namen):
    nummer = {}
    for v in range(len(namen)):
        nummer[namen[v]] = v
    paare = []
    with open(dateiname, encoding='utf-8') as f:
        for zeile in f:
            zeile = zeile.strip()
            if zeile == '':
                continue
            erste, zweite = zeile.split('", "')       # ""Wien", "Prag"" -> ""Wien" und "Prag""
            paare.append((nummer[erste.strip('"')], nummer[zweite.strip('"')]))
    return paare


namen_k, grenze_k = lies_staedte('beispieldaten/cities-small.txt')
paare_k = lies_partnerschaften('beispieldaten/partnerships-small.txt', namen_k)
print(len(namen_k), 'Städte,', len(paare_k), 'Partnerschaften')
print('erste Stadt:', namen_k[0], grenze_k[0], '  erste Partnerschaft:', namen_k[paare_k[0][0]], '-', namen_k[paare_k[0][1]])
print('Summe der Obergrenzen:', sum(grenze_k))

28 Städte, 54 Partnerschaften
erste Stadt: Sofia 0   erste Partnerschaft: Sofia - Prag
Summe der Obergrenzen: 54


Die Summe der Obergrenzen (54) ist genau so groß wie die Zahl der Partnerschaften (54). Das ist kein Zufall: Jedes Fest wird von genau einer Stadt ausgerichtet, die Obergrenzen reichen also **genau** aus, und **jede** Stadt muss ihre Grenze ganz ausnutzen. Das macht die Aufgabe knifflig, denn es gibt keinen Spielraum.

## Schritt 2: Prüfen, ob eine Verteilung gültig ist

Eine Verteilung ist eine Liste `ausrichter` mit einem Eintrag je Partnerschaft: die Nummer der Stadt, die das Fest ausrichtet. Sie ist gültig, wenn
1. der Ausrichter jeder Partnerschaft eine der beiden beteiligten Städte ist und
2. keine Stadt mehr Feste ausrichtet als ihre Obergrenze.

Diese Prüffunktion ist unabhängig vom Verfahren und überprüft später alle Ergebnisse.

In [ ]:
def ist_gueltig(grenze, paare, ausrichter):
    if len(ausrichter) != len(paare):
        return False
    anzahl = [0] * len(grenze)
    for k in range(len(paare)):
        a, b = paare[k]
        if ausrichter[k] != a and ausrichter[k] != b:
            return False                       # ein Fremder richtet das Fest aus
        anzahl[ausrichter[k]] += 1
    for v in range(len(grenze)):
        if anzahl[v] > grenze[v]:
            return False
    return True


# Die erste Stadt jeder Partnerschaft als Ausrichter: gültig?
start_k = [a for a, b in paare_k]
print(ist_gueltig(grenze_k, paare_k, start_k))
ueberlastet = [namen_k[v] for v in range(28) if start_k.count(v) > grenze_k[v]]
print('Überlastet:', ueberlastet)

False
Überlastet: ['Sofia', 'Ljubljana', 'Wien', 'Lyon', 'Vilnius', 'Frankfurt am Main', 'Leipzig', 'Thessaloniki', 'Esch-sur-Alzette', 'Rotterdam']


Der naive Anfang ist nicht gültig, einige Städte sind überlastet. Das erwarten wir, jetzt kommt das Verbessern.

## Schritt 3: Verbessern durch Umdrehen eines Pfades

Wir halten fest:
- `ausrichter[k]`: wer das Fest zur Partnerschaft `k` ausrichtet,
- `fest[v]`: wie viele Feste die Stadt `v` im Moment ausrichtet,
- `nachbarn[v]`: Liste von Paaren `(k, w)`: die Partnerschaft `k` verbindet `v` mit der Stadt `w`.

Eine Kante, die auf `v` zeigt, ist eine Partnerschaft `k` in `nachbarn[v]` mit `ausrichter[k] == v`. Der andere Endpunkt `w` ist eine Stadt, die ihre Last zu `v` „weitergeben“ könnte: Dreht man die Kante um, richtet `w` das Fest aus, nicht mehr `v`.

**Suche** von der überlasteten Stadt `v` aus mit einer Breitensuche rückwärts: Wir merken uns für jede erreichte Stadt `w`, über welche Kante `k` und von welcher Stadt wir gekommen sind (`vorgaenger[w] = (k, stadt)`). Sobald wir eine Stadt mit `fest[w] < grenze[w]` finden, gehen wir die Vorgänger zurück bis `v` und drehen dabei jede Kante um: `ausrichter[k] = w` und dann weiter mit der Stadt, von der wir gekommen sind. Ist keine solche Stadt zu finden, geben wir die Menge der erreichten Städte als Beleg zurück.

In [ ]:
def baue_nachbarn(anzahl_staedte, paare):
    nachbarn = [[] for _ in range(anzahl_staedte)]
    for k in range(len(paare)):
        a, b = paare[k]
        nachbarn[a].append((k, b))
        nachbarn[b].append((k, a))
    return nachbarn


def verteile(grenze, paare):
    """Gibt (ausrichter, anzahl_suchen) zurück oder (None, menge_u), wenn es keine Lösung gibt."""
    n = len(grenze)
    nachbarn = baue_nachbarn(n, paare)
    ausrichter = [a for a, b in paare]            # Anfang: die zuerst genannte Stadt
    fest = [0] * n
    for a in ausrichter:
        fest[a] += 1
    suchen = 0
    for v in range(n):
        while fest[v] > grenze[v]:                # v ist überlastet: verbessern
            suchen += 1
            vorgaenger = {v: None}
            warteschlange = [v]
            gefunden = None
            i = 0
            while i < len(warteschlange) and gefunden is None:
                w = warteschlange[i]
                i += 1
                for k, x in nachbarn[w]:
                    if ausrichter[k] == w and x not in vorgaenger:   # Kante zeigt von x auf w
                        vorgaenger[x] = (k, w)
                        if fest[x] < grenze[x]:
                            gefunden = x
                            break
                        warteschlange.append(x)
            if gefunden is None:
                return None, sorted(vorgaenger)    # keine freie Stadt erreichbar: keine Lösung
            stadt = gefunden
            while vorgaenger[stadt] is not None:   # Pfad zurück bis v, jede Kante umdrehen
                k, naechste = vorgaenger[stadt]
                ausrichter[k] = stadt
                stadt = naechste
            fest[gefunden] += 1
            fest[v] -= 1
    return ausrichter, suchen


ausrichter_k, suchen_k = verteile(grenze_k, paare_k)
print('gültig:', ist_gueltig(grenze_k, paare_k, ausrichter_k), '  Suchen:', suchen_k)

gültig: True   Suchen: 10


Die Verteilung ist gültig. (Der Zähler `suchen` zählt, wie oft ein Pfad gesucht und umgedreht wurde.) Weil die Obergrenzen genau reichen, hat jede Stadt am Ende genau so viele Feste, wie ihre Obergrenze erlaubt, und das ist auch die Tabelle der Lösungshinweise. Die Verteilung selbst ist nicht eindeutig: Die Hinweise zeigen eine andere gültige Verteilung, wichtig ist nur, dass die Prüfung `ist_gueltig` sie annimmt.

Hier die Verteilung für das kleine Beispiel: Für jede Stadt, welche Partnerstädte sie einlädt.

In [ ]:
def zeige_verteilung(namen, paare, ausrichter):
    gaeste = [[] for _ in namen]
    for k in range(len(paare)):
        a, b = paare[k]
        if ausrichter[k] == a:
            gaeste[a].append(namen[b])
        else:
            gaeste[b].append(namen[a])
    for v in range(len(namen)):
        print('%-18s richtet %d aus für: %s' % (namen[v], len(gaeste[v]), ', '.join(gaeste[v])))


zeige_verteilung(namen_k, paare_k, ausrichter_k)

Sofia              richtet 0 aus für: 
Ljubljana          richtet 1 aus für: Wien
Wien               richtet 1 aus für: Prag
Lyon               richtet 1 aus für: Frankfurt am Main
Vilnius            richtet 2 aus für: Ljubljana, Prag
Prag               richtet 2 aus für: Sofia, Frankfurt am Main
Frankfurt am Main  richtet 1 aus für: Leipzig
Brno               richtet 3 aus für: Wien, Leipzig, Poznań
Gdańsk             richtet 4 aus für: Vilnius, Boulogne-sur-Mer, Rotterdam, Nice
Kraków             richtet 2 aus für: Vilnius, Frankfurt am Main
Leipzig            richtet 3 aus für: Lyon, Kraków, Plovdiv
Poznań             richtet 2 aus für: Plovdiv, Hannover
Boulogne-sur-Mer   richtet 0 aus für: 
Nice               richtet 1 aus für: Thessaloniki
Thessaloniki       richtet 2 aus für: Leipzig, Bologna
Esch-sur-Alzette   richtet 4 aus für: Rotterdam, Liège, Turin, Lille
Rotterdam          richtet 3 aus für: Prag, Turin, Liège
Liège              richtet 3 aus für: Kraków, Turin, Köln
Plovd

## Schritt 4: Das große Beispiel

Jetzt das große Beispiel mit 1491 Städten und 2305 Partnerschaften. Wir geben nur Kennzahlen aus.

In [ ]:
namen_g, grenze_g = lies_staedte('beispieldaten/cities.txt')
paare_g = lies_partnerschaften('beispieldaten/partnerships.txt', namen_g)
print(len(namen_g), 'Städte,', len(paare_g), 'Partnerschaften, Summe der Obergrenzen', sum(grenze_g))

ausrichter_g, suchen_g = verteile(grenze_g, paare_g)
print('gültig:', ist_gueltig(grenze_g, paare_g, ausrichter_g), '  Suchen:', suchen_g)

print('Erste fünf Partnerschaften:')
for k in range(5):
    a, b = paare_g[k]
    print('  %s - %s: Fest in %s' % (namen_g[a], namen_g[b], namen_g[ausrichter_g[k]]))
print('höchste Obergrenze:', max(grenze_g), ' Städte mit Obergrenze 0:', grenze_g.count(0))

1491 Städte, 2305 Partnerschaften, Summe der Obergrenzen 2305
gültig: True   Suchen: 791
Erste fünf Partnerschaften:
  Arbon - Binn: Fest in Binn
  Haapsalu - Almere: Fest in Almere
  Haapsalu - Fundão: Fest in Fundão
  Haapsalu - Greve in Chianti: Fest in Greve in Chianti
  Haapsalu - Rendsburg: Fest in Haapsalu
höchste Obergrenze: 22  Städte mit Obergrenze 0: 246


Auch das große Beispiel hat eine gültige Verteilung, und das Verfahren findet sie praktisch sofort. Mit 791 Suchen liegt die Zahl deutlich unter der Obergrenze `m = 2305` unserer Abschätzung: Nicht jede Partnerschaft muss umgedreht werden, und die einzelnen Suchen sind meist kurz. In der Dokumentation würdest du die Verteilung als Datei abgeben (eine Zeile je Partnerschaft). So eine Datei schreiben wir hier mit wenigen Zeilen, in JupyterLite liegt sie im Browser-Speicher.

In [ ]:
with open('verteilung.txt', 'w', encoding='utf-8') as f:
    for k in range(len(paare_g)):
        a, b = paare_g[k]
        f.write('"%s", "%s": %s\n' % (namen_g[a], namen_g[b], namen_g[ausrichter_g[k]]))
print(open('verteilung.txt', encoding='utf-8').read().split('\n')[0])

"Arbon", "Binn": Binn


## Schritt 5: Wenn es keine Lösung gibt

Das Verfahren muss auch „nein“ sagen können, und zwar mit einer **Begründung**. Findet die Suche von einer überlasteten Stadt `v` aus keine Stadt mit freiem Platz, haben wir die Menge `U` aller Städte gefunden, von denen aus man `v` erreicht. Dann gilt:

- Jede Partnerschaft mit einem Endpunkt in `U` und einem außerhalb zeigt aus `U` hinaus. Sonst läge der Endpunkt außerhalb ja auch in `U`. Deshalb können wir sie umdrehen und die Städte in `U` damit nur belasten, nicht entlasten.
- Anders gesagt: Die Partnerschaften **innerhalb** von `U` richtet am Ende immer eine Stadt aus `U` aus. Davon gibt es zu viele, wenn gilt: *Zahl der Partnerschaften innerhalb von `U` > Summe der Obergrenzen in `U`*.

Genau das gilt tatsächlich: Alle Städte in `U` sind ausgelastet (sonst hätte die Suche sie gefunden), `v` ist sogar überlastet. Wir bauen das als Prüfung ein und testen mit einem Beispiel: In einem Dreieck aus A, B, C mit Obergrenzen 1, 1, 0 gibt es 3 Feste, aber nur 2 Plätze.

In [ ]:
def kanten_innen(paare, menge):
    anzahl = 0
    for a, b in paare:
        if a in menge and b in menge:
            anzahl += 1
    return anzahl


def beleg(grenze, paare, menge):
    """Wahr, wenn die Städte der Menge untereinander mehr Partnerschaften haben, als sie Feste ausrichten dürfen."""
    return kanten_innen(paare, menge) > sum(grenze[v] for v in menge)


grenze_d = [1, 1, 0]                              # A, B, C
paare_d = [(0, 1), (1, 2), (0, 2)]
ergebnis, menge_u = verteile(grenze_d, paare_d)
print('Verteilung:', ergebnis, '  Menge U:', menge_u)
print('Beleg stimmt:', beleg(grenze_d, paare_d, set(menge_u)))
print('Partnerschaften innen:', kanten_innen(paare_d, set(menge_u)), ' Summe Obergrenzen:', sum(grenze_d[v] for v in menge_u))

Verteilung: None   Menge U: [0, 1, 2]
Beleg stimmt: True
Partnerschaften innen: 3  Summe Obergrenzen: 2


Auch für das kleine Beispiel aus der Aufgabe können wir „nein“ erzeugen. Die sechs Städte Esch-sur-Alzette, Rotterdam, Liège, Köln, Lille und Turin sind fast alle miteinander verpartnert (15 Partnerschaften untereinander), ihre Obergrenzen betragen zusammen 19. Wir senken Esch-sur-Alzette um 3 und Köln um 2: Dann bleiben 14 Plätze für 15 Feste. Damit die Summe 54 bleibt, erhöhen wir die Obergrenzen von Sofia, Boulogne-sur-Mer, Cork, Coventry und Valencia um je 1.

In [ ]:
grenze_k2 = list(grenze_k)
for stadt, aenderung in [('Esch-sur-Alzette', -3), ('Köln', -2), ('Sofia', 1), ('Boulogne-sur-Mer', 1),
                         ('Cork', 1), ('Coventry', 1), ('Valencia', 1)]:
    grenze_k2[namen_k.index(stadt)] += aenderung
print('Summe der Obergrenzen:', sum(grenze_k2))
ergebnis_k2, menge_k2 = verteile(grenze_k2, paare_k)
if ergebnis_k2 is None:
    print('keine Lösung. Menge U hat', len(menge_k2), 'Städte, darunter', ', '.join(namen_k[v] for v in menge_k2[:4]), '…')
    print('Partnerschaften innerhalb von U:', kanten_innen(paare_k, set(menge_k2)), ' Summe der Obergrenzen:', sum(grenze_k2[v] for v in menge_k2))
else:
    print('Lösung gefunden, gültig:', ist_gueltig(grenze_k2, paare_k, ergebnis_k2))

Summe der Obergrenzen: 54
keine Lösung. Menge U hat 19 Städte, darunter Ljubljana, Wien, Lyon, Vilnius …
Partnerschaften innerhalb von U: 39  Summe der Obergrenzen: 37


Die Menge `U` ist größer als die sechs Städte, die wir verändert haben: Die Suche läuft vom überlasteten Knoten rückwärts und nimmt alles mit, was in der Kette hängt. Das macht nichts, denn der Beleg bleibt richtig (39 Partnerschaften innerhalb von `U`, aber nur 37 Plätze). Er ist nur nicht der kleinste mögliche.

**Zur Probe** vergleichen wir das Verfahren an vielen kleinen Zufallsgraphen mit einem Brute-Force-Verfahren, das alle Richtungen der Kanten durchprobiert (`2^m` Möglichkeiten). Wir prüfen:
- Das Verfahren sagt genau dann „Lösung“, wenn Brute Force eine findet.
- Eine gefundene Verteilung ist gültig.
- Bei „keine Lösung“ stimmt der Beleg.

In [ ]:
import itertools
import random


def brute_force(grenze, paare):
    for wahl in itertools.product([0, 1], repeat=len(paare)):
        anzahl = [0] * len(grenze)
        for k in range(len(paare)):
            anzahl[paare[k][wahl[k]]] += 1
        if all(anzahl[v] <= grenze[v] for v in range(len(grenze))):
            return True
    return False


random.seed(30)
ja = 0
nein = 0
for versuch in range(400):
    n = random.randint(2, 6)
    m = random.randint(1, 10)
    paare_z = []
    for _ in range(m):
        a = random.randrange(n)
        b = random.randrange(n)
        if a != b:
            paare_z.append((a, b))                 # doppelte Partnerschaften sind erlaubt: Test der Hinweise
    grenze_z = [random.randint(0, 3) for _ in range(n)]
    ergebnis_z, info = verteile(grenze_z, paare_z)
    moeglich = brute_force(grenze_z, paare_z)
    if ergebnis_z is None:
        assert not moeglich and beleg(grenze_z, paare_z, set(info)), (grenze_z, paare_z)
        nein += 1
    else:
        assert moeglich and ist_gueltig(grenze_z, paare_z, ergebnis_z), (grenze_z, paare_z)
        ja += 1
print('400 Zufallsgraphen: Lösung gefunden', ja, 'mal, keine Lösung', nein, 'mal, alle Antworten stimmen mit Brute Force überein')

400 Zufallsgraphen: Lösung gefunden 266 mal, keine Lösung 134 mal, alle Antworten stimmen mit Brute Force überein


Zusätzlich ein Test mit größeren, **garantiert lösbaren** Graphen: Wir würfeln einen Zufallsgraphen mit 200 Städten und 500 Partnerschaften, würfeln für jede einen Ausrichter und setzen die Obergrenzen genau auf die gezählten Feste (dann ist die Summe wieder gleich der Zahl der Partnerschaften, wie in der Aufgabe). Das Verfahren muss immer eine Lösung finden. Ein einziges Fest mehr als Plätze (Obergrenze einer Stadt um 1 senken, ohne an anderer Stelle eins zu erhöhen) muss dagegen immer „keine Lösung“ ergeben.

In [ ]:
random.seed(7)
alle_gefunden = True
alle_abgelehnt = True
suchen_summe = 0
for versuch in range(50):
    n = 200
    m = 500
    paare_t = []
    while len(paare_t) < m:
        a = random.randrange(n)
        b = random.randrange(n)
        if a != b:
            paare_t.append((a, b))
    grenze_t = [0] * n
    for a, b in paare_t:
        grenze_t[random.choice([a, b])] += 1
    erg, suchen_t = verteile(grenze_t, paare_t)
    if erg is None or not ist_gueltig(grenze_t, paare_t, erg):
        alle_gefunden = False
    else:
        suchen_summe += suchen_t
    v = random.choice([v for v in range(n) if grenze_t[v] > 0])
    grenze_t[v] -= 1                                # jetzt ein Platz zu wenig
    erg, _ = verteile(grenze_t, paare_t)
    if erg is not None:
        alle_abgelehnt = False
print('immer gelöst:', alle_gefunden, '  mit einem Platz zu wenig immer abgelehnt:', alle_abgelehnt)

immer gelöst: True   mit einem Platz zu wenig immer abgelehnt: True


## Warum nicht einfach der Reihe nach? Ein Greedy-Verfahren und seine Schwäche

Der naheliegende Gedanke: Gehe die Partnerschaften der Reihe nach durch und gib das Fest der Stadt, die noch mehr freie Plätze hat (bei Gleichstand der ersten). Das geht schnell, hat aber eine Schwäche: Eine einmal getroffene Entscheidung wird nie korrigiert. Die Entscheidung kann später genau den Platz verbrauchen, den eine andere Partnerschaft dringend braucht. Wir probieren es an beiden Beispielen.

In [ ]:
def greedy(grenze, paare):
    frei = list(grenze)
    ausrichter = []
    for a, b in paare:
        if frei[b] > frei[a]:
            a = b                                   # die Stadt mit mehr freien Plätzen nimmt das Fest
        if frei[a] <= 0:
            return None                             # beide voll: aufgeben
        frei[a] -= 1
        ausrichter.append(a)
    return ausrichter


print('klein:', 'Lösung' if greedy(grenze_k, paare_k) else 'versagt')
print('groß: ', 'Lösung' if greedy(grenze_g, paare_g) else 'versagt')
versagt = 0
for versuch in range(50):
    n = 200
    m = 500
    paare_t = []
    while len(paare_t) < m:
        a = random.randrange(n)
        b = random.randrange(n)
        if a != b:
            paare_t.append((a, b))
    grenze_t = [0] * n
    for a, b in paare_t:
        grenze_t[random.choice([a, b])] += 1       # lösbar
    if greedy(grenze_t, paare_t) is None:
        versagt += 1
print('Zufallsgraphen (alle lösbar), bei denen Greedy versagt:', versagt, 'von 50')

klein: versagt
groß:  versagt
Zufallsgraphen (alle lösbar), bei denen Greedy versagt: 50 von 50


Der Greedy-Ansatz versagt bei beiden Beispielen und bei allen 50 Zufallsgraphen, obwohl sie alle lösbar sind: Wenn die Obergrenzen genau ausreichen, gibt es keinen Spielraum für eine falsche frühe Entscheidung. Sein „versagt“ bedeutet also nicht „es gibt keine Lösung“, und man darf es nicht so deuten. Unser Verfahren korrigiert frühere Entscheidungen durch das Umdrehen von Pfaden und ist deshalb **vollständig**: Es findet eine Verteilung, wenn es eine gibt, und sonst einen Beleg.

## Für Neugierige

- **Maximaler Fluss.** Die Hinweise zeigen als ersten Ansatz die Rückführung auf einen maximalen Fluss: Quelle → jede Partnerschaft (Kapazität 1) → beide Städte → Senke (Kapazität = Obergrenze). Gibt es einen Fluss, der alle Partnerschaften transportiert, ist die Aufgabe lösbar, und die Kantenrichtung liest man am Fluss ab. Das Umdrehen von Pfaden in unserem Verfahren ist genau der **Ford-Fulkerson-Algorithmus** (bei Breitensuche heißt er Edmonds-Karp) in dieser Sprache.
- **Laufzeit.** Wir haben höchstens `m` Verbesserungen mit je einer Suche in `O(m)`: insgesamt `O(m²)`. Schnellere Flussverfahren (z. B. Dinic) schaffen deutlich weniger, aber für 2305 Partnerschaften ist unser Verfahren schon schnell genug.
- **Der Beleg ist der Min-Cut.** Die Menge `U` im „nein“-Fall ist ein minimaler Schnitt des Netzwerks. Der Satz, dass maximaler Fluss = minimaler Schnitt, steckt hinter unserer Begründung. Er heißt Satz von Max-Flow Min-Cut.

## Zum Schluss: dasselbe mit MiniZinc

Die Aufgabe ist ein **Zuordnungsproblem** und passt gut zu MiniZinc: Wir beschreiben nur, was gelten soll, und der Solver sucht.

Für jede Partnerschaft `e` gibt es eine Variable `x[e]`: 1 heißt, die erste Stadt `a[e]` richtet das Fest aus, 0 heißt, die zweite Stadt `b[e]`. Für jede Stadt `v` zählen wir die Feste, die sie ausrichtet: die Partnerschaften mit `a[e] == v` und `x[e] = 1` plus die mit `b[e] == v` und `x[e] = 0`. Diese Summe darf `grenze[v]` nicht übersteigen. Die Summen laufen mit `sum(e in 1..m where a[e] == v)(…)` nur über die passenden Partnerschaften, wie in den Arbeitsblättern zu `sum … where`.

```minizinc
int: n;                                  % Anzahl Städte
int: m;                                  % Anzahl Partnerschaften
array[1..n] of int: grenze;
array[1..m] of 1..n: a;                  % erste Stadt der Partnerschaft
array[1..m] of 1..n: b;                  % zweite Stadt
array[1..m] of var 0..1: x;              % 1: a[e] richtet das Fest aus, 0: b[e]

constraint forall(v in 1..n)(
  sum(e in 1..m where a[e] == v)(x[e]) + sum(e in 1..m where b[e] == v)(1 - x[e]) <= grenze[v]
);

solve satisfy;

output ["x = ", show(x), "\n"];
```

Die Daten bekommt MiniZinc als `.dzn`-Datei. Die Python-Zelle `als_dzn` baut sie aus den Listen `grenze` und `paare`. Wir zählen dabei von 1, wie MiniZinc es verlangt.

In [ ]:
def als_dzn(grenze, paare):
    erste = [p[0] + 1 for p in paare]
    zweite = [p[1] + 1 for p in paare]
    return 'n = %d;\nm = %d;\ngrenze = %s;\na = %s;\nb = %s;' % (len(grenze), len(paare), grenze, erste, zweite)


daten_k = als_dzn(grenze_k, paare_k)
print(daten_k[:200], '…')

n = 28;
m = 54;
grenze = [0, 1, 1, 1, 2, 2, 1, 3, 4, 2, 3, 2, 0, 1, 2, 4, 3, 3, 2, 1, 6, 3, 1, 1, 3, 0, 1, 1];
a = [1, 2, 2, 3, 3, 4, 4, 5, 5, 5, 6, 6, 7, 7, 8, 8, 8, 9, 9, 9, 9, 10, 10, 11, 11, 11, 1 …


Für das kleine Beispiel liefert der Solver in einem Augenblick eine Belegung von `x`:

**Ausgabe:**

```
x = [0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1]
----------
```

Ob sie stimmt, prüfen wir wie bei unserem Verfahren: Aus `x` bauen wir die Liste der Ausrichter und lassen `ist_gueltig` urteilen (beim Erzeugen dieses Notebooks war sie gültig, auch beim großen Beispiel). Für die Aufgabe mit den veränderten Obergrenzen aus Schritt 5 meldet MiniZinc `=====UNSATISFIABLE=====`, also „keine Lösung“, und stimmt damit mit unserem Verfahren überein. Einen **Beleg** wie die Menge `U` liefert der Solver allerdings nicht.

**Wie gut ist der Solver?** Das hängt stark vom Solver ab. Beim großen Beispiel (1491 Städte, 2305 Partnerschaften) lief es so (gemessen auf einem normalen Rechner):

| Solver | kleines Beispiel | großes Beispiel |
|---|---|---|
| Gecode | Sekundenbruchteile | nach 30 Sekunden ohne Ergebnis abgebrochen |
| Chuffed | Sekundenbruchteile | nach 30 Sekunden ohne Ergebnis abgebrochen |
| CP-SAT | Sekundenbruchteile | auch nach über 3 Minuten nicht fertig (Prozess abgebrochen) |
| HiGHS | Sekundenbruchteile | wenige Sekunden |

Warum? Gecode und Chuffed probieren Belegungen durch und müssen dabei viel zurücknehmen, weil die Obergrenzen genau reichen und fast jede frühe Entscheidung irgendwann scheitert. **HiGHS** ist ein MIP-Solver, der das Modell als ganzzahliges lineares Programm löst. Dieses Programm ist genau das Flussproblem der Hinweise: Seine LP-Relaxation (ohne die Forderung, dass `x` ganzzahlig ist) liefert von selbst ganzzahlige Lösungen. Deshalb findet HiGHS die Lösung ohne lange Suche. Die Wahl des Solvers entscheidet hier über Sekunden oder gar kein Ergebnis. Unser eigenes Verfahren braucht dafür keinen Solver und ist trotzdem schnell, weil es die Struktur der Aufgabe (verbessernde Pfade) ausnutzt.